# Task Scores with Related Occupations and Work Activities

This notebook computes task-level automation desire and capacity scores, along with related occupations and work activities for each task. Two new columns have been added:

- **related_occupations**: A list of unique occupations related to each task, derived from the domain_worker_desires dataset
- **related_work_activities**: A list of unique work activities related to each task, derived from the domain_worker_desires dataset

The implementation uses robust column detection to handle different dataset variations and automatically detects occupation and work activity columns using candidate lists.

In [1]:
import pandas as pd
from datasets import load_dataset

# Load datasets
domain_worker_desires = load_dataset("SALT-NLP/WORKBank", data_files="worker_data/domain_worker_desires.csv")["train"].to_pandas()
expert_ratings = load_dataset("SALT-NLP/WORKBank", data_files="expert_ratings/expert_rated_technological_capability.csv")["train"].to_pandas()

# Function to detect columns using candidate lists
def find_column(df, candidates):
    """Find first matching column from candidate list"""
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None

# Function to process values into lists of unique strings
def process_to_list(value):
    """Convert value to list of unique strings, handling different input types"""
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        # Already a list, convert elements to strings and remove duplicates
        return sorted(list(set(str(item) for item in value if pd.notna(item))))
    elif isinstance(value, str):
        # Try to evaluate as list literal, otherwise treat as single string
        try:
            eval_value = eval(value)
            if isinstance(eval_value, list):
                return sorted(list(set(str(item) for item in eval_value if pd.notna(item))))
            else:
                return [str(value)] if value.strip() else []
        except:
            return [str(value)] if value.strip() else []
    else:
        return [str(value)]

# Define candidate column names
occupation_candidates = ['Occupation', 'Occupations', 'Worker Occupation', 'Occupation (O*NET-SOC Title)']
activity_candidates = ['Related Work Activities', 'Work Activities', 'Work Activity', 'Related Work Activity', 'Skill (O*NET Work Activity)']

# Find occupation and activity columns
occupation_col = find_column(domain_worker_desires, occupation_candidates)
activity_col = find_column(domain_worker_desires, activity_candidates)

# Create aggregation dict
agg_dict = {}

# Add occupation aggregation if column exists
if occupation_col:
    agg_dict['related_occupations'] = lambda x: sorted(list(set(str(item) for item in x if pd.notna(item))))

# Add activity aggregation if column exists  
if activity_col:
    def aggregate_activities(series):
        all_activities = []
        for value in series:
            activities = process_to_list(value)
            all_activities.extend(activities)
        return sorted(list(set(all_activities)))
    
    agg_dict['related_work_activities'] = aggregate_activities

# Compute related occupations and activities by task
if agg_dict:
    # Create mapping dict with appropriate column names
    rename_dict = {}
    if occupation_col:
        rename_dict[occupation_col] = 'related_occupations'
    if activity_col:
        rename_dict[activity_col] = 'related_work_activities'
    
    # Aggregate by task
    task_related = domain_worker_desires.groupby('Task').agg({
        col: agg_dict[new_name] for col, new_name in rename_dict.items()
    }).rename(columns=rename_dict)
else:
    # Create empty DataFrame if no columns found
    task_related = pd.DataFrame(index=domain_worker_desires['Task'].unique())
    task_related.index.name = 'Task'

# Add empty columns if not found
if 'related_occupations' not in task_related.columns:
    task_related['related_occupations'] = [[] for _ in range(len(task_related))]
if 'related_work_activities' not in task_related.columns:
    task_related['related_work_activities'] = [[] for _ in range(len(task_related))]

# Reset index to make Task a column
task_related = task_related.reset_index()

# Compute automation scores
desire_by_task = domain_worker_desires.groupby("Task")["Automation Desire Rating"].mean().rename("automation_desire")
capacity_by_task = expert_ratings.groupby("Task")["Automation Capacity Rating"].mean().rename("automation_capacity")

# Combine all task data with proper column ordering
task_scores = pd.concat([desire_by_task, capacity_by_task], axis=1).dropna().reset_index()

# Merge with related data, ensuring column order: Task, related_occupations, related_work_activities, automation_desire, automation_capacity
task_scores = task_scores.merge(task_related, on='Task', how='left')

# Reorder columns to put related columns right after Task
cols = ['Task', 'related_occupations', 'related_work_activities', 'automation_desire', 'automation_capacity']
# Add any additional columns that might exist
additional_cols = [col for col in task_scores.columns if col not in cols]
task_scores = task_scores[cols + additional_cols]

# Fill NaN values with empty lists for related columns
task_scores['related_occupations'] = task_scores['related_occupations'].apply(lambda x: x if isinstance(x, list) else [])
task_scores['related_work_activities'] = task_scores['related_work_activities'].apply(lambda x: x if isinstance(x, list) else [])

/Users/aperio/Projects/workbank/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#sort by "automation_capacity" desc
task_scores = task_scores.sort_values(by="automation_capacity", ascending=False)
# export top 50 as csv
task_scores.to_csv("./dist/task_scores.csv", index=False)